# Working with typical periods

`tsam.aggregate()` hands back more than a few typical days: it also returns the bookkeeping
that links every typical day back to the original calendar. This guide shows how to read and
use those links — inspect the mapping, feed a downstream model, map its results back, and see
how the typical days relate to the original series (the information seasonal-storage
formulations rely on).

We reuse the tutorial's setup: six weeks of hourly data aggregated to six typical days.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}

result = tsam.aggregate(data, n_clusters=6, period_duration="1D")
print(
    "typical days:",
    result.n_clusters,
    " original days:",
    len(result.cluster_assignments),
)

## What you get back

Three pieces describe the aggregation, and together they let you move between the typical
periods and the original calendar:

* **`cluster_representatives`** — the typical day-shapes (what a downstream model optimises over).
* **`cluster_counts`** — how many real days each typical day stands for (its weight).
* **`cluster_assignments`** — for each original day, in calendar order, which typical day represents it.

In [ ]:
print("cluster_counts:", result.cluster_counts)
print("cluster_assignments (one per original day, in order):")
print(result.cluster_assignments)
result.cluster_representatives.head()

## See the mapping on the calendar

Each real day is shaded with the colour of the typical day standing in for it. A run of one
colour is a stretch of consecutive days that share a representative:

In [ ]:
result.plot.clusters_over_time(
    columns=["Load"],
    units=UNITS,
    title="Which typical day represents each real day",
)

The same mapping is available as a per-timestep table — handy for joining onto the original
series or saving to CSV:

In [ ]:
result.assignments.head()

## Members of a typical day

The assignment also runs the other way: every typical day has a set of **member** days. Pick
a cluster to list its members and see them with the representative highlighted:

In [ ]:
cluster = 0
members = np.where(result.cluster_assignments == cluster)[0]
print(
    f"Typical day {cluster} represents {len(members)} real days, at positions {members.tolist()}"
)
result.plot.cluster_members(columns=["Load"], clusters=[cluster], units=UNITS)

## Round-trip: map model results back to the full timeline

The usual workflow is to optimise on the six typical days, then expand the per-typical-day
results back over the original six weeks. `disaggregate()` does the expansion — it replaces
each original day with the row of its representative. Disaggregating the representatives
themselves reproduces the reconstruction exactly, which shows the mapping at work:

In [ ]:
back = result.disaggregate(result.cluster_representatives)
print(
    "disaggregate(representatives) == reconstructed:",
    np.allclose(back.values, result.reconstructed.values),
)
print("shape:", result.cluster_representatives.shape, "->", back.shape)

Any DataFrame shaped like `cluster_representatives` maps back the same way — for example a
per-typical-day decision your model produced. Here we stand in a made-up dispatch (half the
representative Load) and expand it to every real day:

In [ ]:
# stand-in for a model output: one profile per typical day
model_output = result.cluster_representatives[["Load"]] * 0.5
full = result.disaggregate(model_output)

px.line(
    full.reset_index(names="time"),
    x="time",
    y="Load",
    title="A per-typical-day result expanded to the full six weeks",
    labels={"Load": f"dispatch [{UNITS['Load']}]", "time": "time"},
)

## Linking periods in time: inter-period storage

`cluster_assignments` is an **ordered** vector — it keeps the calendar sequence of which
typical day stands in for each real day. That sequence is exactly what seasonal-storage
formulations need.

> **Why the order matters (Kotzur et al. 2018).** Operating decisions are modelled on the
> few typical days, but a seasonal store's state of charge has to be tracked across the
> *real* chronology of the year. Kotzur et al. (2018) split the storage state into an
> *intra-period* part (within a typical day) and an *inter-period* part that carries over
> between the original days in sequence — using precisely the typical-day assignment
> *k = f(day)* that tsam returns here. tsam provides the linkage (the ordered assignments);
> the inter-period constraints themselves live in your optimisation model, not in tsam.

The plot below *is* that sequence: every original day, labelled with the typical day it maps
to. Reading it left to right is the order an inter-period storage balance would follow:

In [ ]:
palette = px.colors.qualitative.Plotly
cluster_ids = sorted({int(c) for c in result.cluster_assignments})
color_map = {str(c): palette[i % len(palette)] for i, c in enumerate(cluster_ids)}

seq = pd.DataFrame(
    {
        "day": np.arange(len(result.cluster_assignments)),
        "typical_day": [str(int(c)) for c in result.cluster_assignments],
    }
)
fig = px.scatter(
    seq,
    x="day",
    y="typical_day",
    color="typical_day",
    color_discrete_map=color_map,
    category_orders={"typical_day": [str(c) for c in cluster_ids]},
    title="Sequence of typical days across the six weeks (k = f(day))",
)
fig.update_traces(marker={"size": 11})
fig.update_layout(
    xaxis_title="original day (calendar order)",
    yaxis_title="typical day",
    showlegend=False,
)
fig.show()

## Where to go next

* [Optimization workflow](optimization_workflow.ipynb) — the full hand-off to a downstream
  model (representatives, counts, weights) and reusing a clustering across datasets.
* [Extreme periods](../how_it_works/04_extreme_periods.ipynb) — make sure the peak day survives
  before you rely on the mapping for capacity sizing.
* [Further reading](../../further-reading.md) — Kotzur et al. (2018), the inter-period storage
  formulation that consumes the typical-day sequence.